##  02_features_ml 
### Construção da base agregada para modelagem

ENEM MG 2021–2023 — Engenharia de atributos e agregação para modelagem
Objetivo

Este notebook constrói uma base agregada voltada à modelagem preditiva, utilizando os microdados consolidados do ENEM para Minas Gerais (2021–2023).

A proposta é transformar observações individuais de candidatos em unidades estruturais comparáveis, representando perfis socioeducacionais definidos por:

* ano do exame
* faixa de renda familiar (salários mínimos)
* tipo de escola frequentada

Essa agregação permite estimar relações entre condições socioeconômicas médias e desempenho educacional médio, reduzindo o ruído individual presente nas observações micro.

Entradas

DADOS_MG_21_23_NUM
Base consolidada de Minas Gerais (2021–2023) em formato numérico.

Saídas

DADOS_AGG_MG_ML
Base agregada utilizada nas etapas de modelagem preditiva.

### 1) Ambiente e imports

In [1]:
import sys
from pathlib import Path

# Permite importar o pacote `src/` a partir do diretório do projeto.
ROOT_PATH = Path().resolve().parents[1]  # notebooks/00_preprocessamento -> projeto
if str(ROOT_PATH) not in sys.path:
    sys.path.append(str(ROOT_PATH))

import re
import numpy as np
import pandas as pd


from src.config import (
    DADOS_MG_21_23_NUM, 
    DADOS_AGG_MG_ML
    
)


#pandas configurando para mostrar todas as linhas e colunas
pd.set_option ('display.max_columns', None)
#configurando pandas para não mostrar notação científica
pd.set_option('display.float_format', lambda x: '%.2f' % x)




### 2) Leitura da base consolidada

A base utilizada neste notebook corresponde ao recorte de Minas Gerais entre 2021 e 2023, já previamente tratado e convertido para uma representação numérica das variáveis ordinais.

In [2]:
df = pd.read_parquet( DADOS_MG_21_23_NUM)

df.head(3)

,faixa_etaria,sexo,estado_civil,cor_raca,escola,uf,nota_cn,nota_ch,nota_lc,nota_mt,lingua,nota_redacao,escolaridade_pai,escolaridade_mae,ocup_pai,ocup_mae,n_pessoas_resd,sal_min,emp_domst,gelad,lv_rp,tv,cel,comptdr,indice_consumo,renda_media,nota_media,regiao,ano,escola_num,escolaridade_pais_media,ocup_pais_media
0,20-25,feminino,solteiro,branca,não informada,MG,NaN,574.60,472.60,NaN,espanhol,760.00,4,4,2,2,3,1 a 3,0,1,1,1,1,1,0.18,2.00,602.40,Metrop. de Belo Horizonte,2021,1,4.00,2.00
1,20-25,feminino,solteiro,negra,pública,MG,487.40,476.50,450.70,493.40,inglês,520.00,2,2,2,2,2,1 a 3,0,1,0,1,1,0,0.15,2.00,485.60,Metrop. de Belo Horizonte,2021,0,2.00,2.00
2,20-25,masculino,solteiro,negra,não informada,MG,582.30,664.30,576.00,528.70,inglês,580.00,2,3,2,2,3,1 a 3,5,1,1,3,3,2,0.32,2.00,586.26,Metrop. de Belo Horizonte,2021,1,2.50,2.00


### 3) Estrutura da base

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 996185 entries, 0 to 996184
Data columns (total 32 columns):
 #   Column                   Non-Null Count   Dtype   
---  ------                   --------------   -----   
 0   faixa_etaria             996185 non-null  category
 1   sexo                     996185 non-null  category
 2   estado_civil             996185 non-null  category
 3   cor_raca                 996185 non-null  category
 4   escola                   996185 non-null  category
 5   uf                       996185 non-null  category
 6   nota_cn                  686584 non-null  float32 
 7   nota_ch                  720293 non-null  float32 
 8   nota_lc                  720293 non-null  float32 
 9   nota_mt                  686584 non-null  float32 
 10  lingua                   996185 non-null  category
 11  nota_redacao             720293 non-null  float32 
 12  escolaridade_pai         996185 non-null  Int16   
 13  escolaridade_mae         996185 non-null  In

A base contém aproximadamente 1 milhão de observações, com variáveis relacionadas a:

* características demográficas
* contexto familiar
* infraestrutura domiciliar
* renda aproximada
* notas do ENEM

As variáveis de escolaridade e ocupação dos pais já foram previamente convertidas para representações numéricas ordinais, permitindo análises quantitativas.

### 4) Estatísticas descritivas

In [4]:
df.describe()

,nota_cn,nota_ch,nota_lc,nota_mt,nota_redacao,escolaridade_pai,escolaridade_mae,ocup_pai,ocup_mae,n_pessoas_resd,emp_domst,gelad,lv_rp,tv,cel,comptdr,indice_consumo,renda_media,nota_media,escola_num,escolaridade_pais_media,ocup_pais_media
count,686584.00,720293.00,720293.00,686584.00,720293.00,996185.00,996185.00,996185.00,996185.00,996185.00,996185.00,996185.00,996185.00,996185.00,996185.00,996185.00,996185.00,996185.00,723122.00,996185.00,996185.00,996185.00
mean,511.39,542.36,528.78,567.68,652.34,2.49,2.93,2.56,2.52,3.61,0.25,1.06,0.74,1.41,2.72,0.94,0.29,3.46,559.21,0.80,2.71,2.54
std,82.21,87.30,74.53,123.43,201.07,1.19,1.14,1.44,1.34,1.26,0.96,0.29,0.47,0.83,1.04,0.91,0.12,3.53,94.41,0.52,1.00,1.17
min,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.50,0.00,0.00,0.00,0.00
25%,452.90,487.00,482.70,470.50,540.00,2.00,2.00,1.00,2.00,3.00,0.00,1.00,0.00,1.00,2.00,0.00,0.19,2.00,496.00,0.00,2.00,2.00
50%,507.40,548.60,534.70,564.35,640.00,2.00,3.00,3.00,2.00,4.00,0.00,1.00,1.00,1.00,3.00,1.00,0.26,2.00,558.14,1.00,2.50,2.50
75%,566.90,602.70,580.80,657.20,800.00,3.00,4.00,4.00,4.00,4.00,0.00,1.00,1.00,2.00,4.00,1.00,0.36,4.00,625.16,1.00,3.50,3.50
max,868.70,846.90,820.80,985.70,1000.00,5.00,5.00,5.00,5.00,20.00,5.00,4.00,4.00,4.00,4.00,4.00,1.00,22.00,851.04,2.00,5.00,5.00


In [5]:
df.describe(exclude='number')

,faixa_etaria,sexo,estado_civil,cor_raca,escola,uf,lingua,sal_min,regiao,ano
count,996185,996185,996185,996185,996185,996185,996185,996185,996185,996185
unique,6,2,5,5,3,1,2,7,13,3
top,até 19,feminino,solteiro,negra,não informada,MG,inglês,1 a 3,Metrop. de Belo Horizonte,2023
freq,608083,624307,894743,513453,690166,996185,649231,688725,346145,358575


Essas estatísticas fornecem uma visão geral da distribuição das variáveis socioeconômicas e educacionais presentes na base.

### 5) Seleção das variáveis relevantes

Como a variável alvo do modelo será nota_media, as demais notas individuais são removidas para evitar redundância ou vazamento de informação. 

Para evitar multicolinearidade artificial, também removerei indice_consumo que combina variáveis cel e comptdr que já estão no modelo.

In [6]:

colunas_notas_dropar=[
    'nota_cn',
    'nota_ch',
    'nota_lc',
    'nota_mt',
    'lingua',
    'nota_redacao',
    'indice_consumo', 
]
df=df.drop(columns=colunas_notas_dropar)
df.head()

,faixa_etaria,sexo,estado_civil,cor_raca,escola,uf,escolaridade_pai,escolaridade_mae,ocup_pai,ocup_mae,n_pessoas_resd,sal_min,emp_domst,gelad,lv_rp,tv,cel,comptdr,renda_media,nota_media,regiao,ano,escola_num,escolaridade_pais_media,ocup_pais_media
0,20-25,feminino,solteiro,branca,não informada,MG,4,4,2,2,3,1 a 3,0,1,1,1,1,1,2.00,602.40,Metrop. de Belo Horizonte,2021,1,4.00,2.00
1,20-25,feminino,solteiro,negra,pública,MG,2,2,2,2,2,1 a 3,0,1,0,1,1,0,2.00,485.60,Metrop. de Belo Horizonte,2021,0,2.00,2.00
2,20-25,masculino,solteiro,negra,não informada,MG,2,3,2,2,3,1 a 3,5,1,1,3,3,2,2.00,586.26,Metrop. de Belo Horizonte,2021,1,2.50,2.00
3,até 19,feminino,solteiro,negra,não informada,MG,2,2,1,2,4,1 a 3,0,1,0,1,2,0,2.00,529.64,Metrop. de Belo Horizonte,2021,1,2.00,1.50
4,26-35,masculino,solteiro,branca,não informada,MG,2,2,0,2,5,1 a 3,0,1,1,1,3,1,2.00,NaN,Sul de Minas,2021,1,2.00,1.00


### 6) Tratamento de valores ausentes

Verificação dos valores ausentes:

In [7]:
df.isna().sum()

faixa_etaria                    0
sexo                            0
estado_civil                    0
cor_raca                        0
escola                          0
uf                              0
escolaridade_pai                0
escolaridade_mae                0
ocup_pai                        0
ocup_mae                        0
n_pessoas_resd                  0
sal_min                         0
emp_domst                       0
gelad                           0
lv_rp                           0
tv                              0
cel                             0
comptdr                         0
renda_media                     0
nota_media                 273063
regiao                          0
ano                             0
escola_num                      0
escolaridade_pais_media         0
ocup_pais_media                 0
dtype: int64

Os valores ausentes ocorrem apenas na variável nota_media, refletindo candidatos que não tiveram notas válidas em todas as provas.

Como o objetivo é construir um modelo preditivo do desempenho, os registros sem nota média são removidos.

In [8]:
df=df.dropna()
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 723122 entries, 0 to 996184
Data columns (total 25 columns):
 #   Column                   Non-Null Count   Dtype   
---  ------                   --------------   -----   
 0   faixa_etaria             723122 non-null  category
 1   sexo                     723122 non-null  category
 2   estado_civil             723122 non-null  category
 3   cor_raca                 723122 non-null  category
 4   escola                   723122 non-null  category
 5   uf                       723122 non-null  category
 6   escolaridade_pai         723122 non-null  Int16   
 7   escolaridade_mae         723122 non-null  Int16   
 8   ocup_pai                 723122 non-null  Int16   
 9   ocup_mae                 723122 non-null  Int16   
 10  n_pessoas_resd           723122 non-null  Int8    
 11  sal_min                  723122 non-null  category
 12  emp_domst                723122 non-null  Int8    
 13  gelad                    723122 non-null  Int8   

### 7) Construção de variáveis familiares agregadas

Para sintetizar o capital educacional e ocupacional da família, são criadas duas variáveis médias:

* escolaridade_pais_media
* ocup_pais_media

In [9]:
df['escolaridade_pais_media'] = (df['escolaridade_pai'] + df['escolaridade_mae']) / 2
df['ocup_pais_media'] = (df['ocup_mae'] + df['ocup_pai']) / 2

In [10]:
df['regiao'].value_counts()

regiao
Metrop. de Belo Horizonte     245594
Triâng. Min. e Alto Paran.     86630
Sul de Minas                   86144
Norte de Minas                 65745
Sudoeste de Minas              54204
Vale do Rio Doce               48222
Oeste de Minas                 32160
Zona da Mata                   26092
Vale do Jequitinhonha          23699
Campo das Vertentes            17546
Centro de Minas                12633
Noroeste de Minas              12578
Vale do Mucuri                 11875
Name: count, dtype: int64

Essas variáveis capturam o nível médio de capital educacional e ocupacional familiar.

### 8) Padronização de nomes das variáveis

Algumas colunas são renomeadas para facilitar a interpretação dos coeficientes durante a análise do modelo.

In [11]:
#ajustar os nomes das variáveis para facilitar a leitura quando os coeficientes forem avaliados
df=df.rename(columns=
    {
    'ano': 'Ano',
    'sal_min': 'SalMin',
    'escola':'Escola',
    'raça': 'Raça',
    'sexo': 'Sexo',
    'faixa_etaria':'FaixaEtaria',
    'estado_civil': 'EstadoCivil',
    'regiao': 'Regiao',
    }
)

### 9) Agregação socioeducacional

Os dados são agregados por:

* ano
* faixa de renda (SalMin)
* tipo de escola
* regiao

Essas agregações criam unidades analíticas que representam perfis socioeducacionais médios. A opção sem região será usada para trabalhar umaq opção de modelo mais genérico

In [27]:
cols_agg = ['Ano', 'SalMin', 'Escola',]

for col in cols_agg:
    if df[col].dtype.name == 'category':
        df[col] = df[col].cat.remove_unused_categories()

        


#### Construção da base agregada

In [28]:
df_agg = df.groupby(cols_agg, observed=True, dropna=False).agg(
    Participantes=("Ano", "count"),
    Cel=("cel", 'mean'),  
    Comptdr=("comptdr", 'mean'),
    PessoasResd=("n_pessoas_resd",'mean'),  
    NotaMedia=('nota_media','mean'),
    OcupPaisMedia=('ocup_pais_media', 'mean'),
    EscolaridadePaisMedia=('escolaridade_pais_media', 'mean'),

    
).reset_index()



### 10) Filtragem de grupos pequenos

Para evitar estimativas instáveis, grupos com poucos participantes são removidos.

A variável Participantes representa o tamanho amostral de cada grupo agregado e é preservada na base final para permitir ponderação estatística na etapa de modelagem.

In [29]:
MIN_PARTICIPANTES = 10
df_agg = df_agg.query("Participantes >= @MIN_PARTICIPANTES")

df_agg.info()

<class 'pandas.core.frame.DataFrame'>
Index: 63 entries, 0 to 62
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype   
---  ------                 --------------  -----   
 0   Ano                    63 non-null     object  
 1   SalMin                 63 non-null     category
 2   Escola                 63 non-null     category
 3   Participantes          63 non-null     int64   
 4   Cel                    63 non-null     Float64 
 5   Comptdr                63 non-null     Float64 
 6   PessoasResd            63 non-null     Float64 
 7   NotaMedia              63 non-null     float32 
 8   OcupPaisMedia          63 non-null     Float64 
 9   EscolaridadePaisMedia  63 non-null     Float64 
dtypes: Float64(5), category(2), float32(1), int64(1), object(1)
memory usage: 5.1+ KB


### 13) Diagnóstico da base agregada

Antes da exportação da base final utilizada na modelagem, é útil apresentar um breve diagnóstico estrutural da base agregada.

Esse diagnóstico permite verificar:

número de perfis socioeducacionais distintos
tamanho médio dos grupos
distribuição do número de participantes por grupo

Essas informações ajudam a avaliar a estabilidade das médias utilizadas na modelagem.

In [30]:
print("Perfis socioeducacionais (linhas da base):", len(df_agg))
print("Participantes totais representados:", df_agg["Participantes"].sum())
print("Participantes médios por grupo:", round(df_agg["Participantes"].mean(),2))
print("Participantes mediana por grupo:", round(df_agg["Participantes"].median(),2))

Perfis socioeducacionais (linhas da base): 63
Participantes totais representados: 723122
Participantes médios por grupo: 11478.13
Participantes mediana por grupo: 3250.0


### Preparação final da base de modelagem

Após a verificação estrutural da base agregada, os dados estão prontos para serem exportados e utilizados na etapa de modelagem.

Cada linha da base representa um perfil socioeducacional médio, com medidas agregadas de condições familiares e desempenho educacional.

### 14) Exportação da base de modelagem

In [31]:
df_agg.to_parquet(DADOS_AGG_MG_ML, index=False)

### 15) Interpretação da base agregada

A base final df_agg representa perfis socioeducacionais específicos, contendo:

Participantes → tamanho do grupo
NotaMedia → desempenho médio do perfil
variáveis familiares médias
indicadores de infraestrutura domiciliar
distribuição regional dos participantes (proporções)

### Uso da variável Participantes

A variável Participantes representa o número de candidatos que compõem cada grupo agregado (definido por Ano × SalMin × Escola).

Ela é preservada na base final por dois motivos principais:

permitir a análise da distribuição do tamanho dos grupos
possibilitar o uso de ponderação estatística na etapa de modelagem

Essa ponderação permite que grupos maiores contribuam com estimativas mais estáveis.

### Interpretação metodológica

Os dados foram agregados por:

ano do exame
faixa de renda
tipo de escola

A variável região é incorporada indiretamente por meio da composição proporcional dos participantes em cada região dentro dos grupos agregados.

Essa abordagem permite:

reduzir o ruído individual presente nos microdados
preservar a heterogeneidade geográfica sem fragmentar excessivamente a base
estimar relações entre condições socioeconômicas médias e desempenho educacional médio

Assim, o modelo resultante deve ser interpretado como um modelo explicativo-estrutural com capacidade preditiva auxiliar.

#### Boas práticas adotadas

* observed=True
    -> Evita a criação de combinações artificiais de categorias.
* remoção de categorias não utilizadas
    -> Reduz dimensionalidade e melhora eficiência.
* inclusão do tamanho do grupo (Participantes)
    -> Essencial para interpretar a relevância estatística dos perfis.
* exclusão de grupos muito pequenos
    -> Evita que resultados sejam dominados por observações isoladas.

### Interpretação metodológica

Os dados foram agregados por ano, renda familiar e tipo de escola, com o objetivo de transformar observações individuais em unidades estruturais comparáveis.

Essa abordagem permite:

* reduzir o ruído individual presente nos microdados
* estimar relações entre condições socioeconômicas médias e desempenho educacional médio
* interpretar resultados sob uma perspectiva estrutural de desigualdade educacional

Assim, o modelo resultante deve ser interpretado como um modelo explicativo-estrutural com capacidade preditiva auxiliar, voltado a compreender padrões socioeducacionais, e não apenas prever o desempenho individual de candidatos.

### Limitações metodológicas da agregação

A construção da base agregada melhora a interpretabilidade estrutural do problema, mas introduz algumas limitações importantes que devem ser explicitadas.

##### 1. Perda de variabilidade individual

Ao agregar candidatos em grupos definidos por Ano × SalMin × Escola, informações individuais deixam de ser observadas diretamente.


##### 2. Dependência da escolha das chaves de agregação

Os resultados do modelo dependem da forma como os grupos são definidos.

Neste projeto, optou-se por agregar por:

ano do exame
faixa de renda
tipo de escola

A dimensão regional é incorporada por meio de proporções, evitando segmentação excessiva.

##### 3. Risco de instabilidade em grupos pequenos

GGrupos com poucos participantes tendem a apresentar médias mais sensíveis a flutuações amostrais.

Por isso, foi adotado um filtro mínimo de participantes por grupo.

##### 4. Heterogeneidade interna capturada parcialmente

Mesmo após a agregação, um grupo pode conter participantes de diferentes regiões.

Essa heterogeneidade é parcialmente capturada pela distribuição proporcional regional, mas outras dimensões não observadas podem permanecer.

##### 5. Interpretação do modelo

O modelo resultante deve ser interpretado como um modelo de padrões estruturais médios, no qual:

* as variáveis socioeconômicas explicam diferenças médias entre perfis
* a dimensão regional atua como um fator contextual contínuo
* os coeficientes associados às regiões refletem efeitos marginais da composição geográfica dos grupos

O modelo não deve ser interpretado como:

* modelo de causalidade individual
* preditor direto de desempenho individual